# Shapley Value
---

Shapley Value is a concept from cooperative game theory that gives us a principled way to fairly distribute a collective payoff among players. It was introduced in 1953 by Lloyd Shapley, who later received the Nobel Prize in Economics for this and related work. Because of its elegant theoretical properties, the idea has since been adopted far beyond game theory — including in Marketing and Machine Learning, where it now serves as a foundation for explaining how individual features contribute to a model's predictions.

So, let's dig in.

### Problem Setting
In non-cooperative games, players act independently. For example, in individual competitions, the player who runs fastest or jumps highest simply takes the prize. So The attribution is straightforward.

In cooperative games, players must coordinate. Player A might support Player B, Player C might only be called in for penalty shots, and so on — there's specialization. But their ultimate goal is to maximize the team's collective outcome.

Since the total gain in cooperative games is not simply the sum of individual contributions, it's not always obvious how to fairly measure each player's share. But the core intuition is natural: a player's contribution is the *"result with the player"* minus the *"result without the player"*. Crucially, we need to average this difference across all possible game configurations — a player might perform well against weak opponents and poorly against strong ones, or excel in one position but struggle in another. All of that has to be accounted for.

Let's write this down formally.

Let:
- $N$ = the full set of players
- $S \subseteq N$ = a coalition (subset of players)
- $v(S)$ = the payoff the coalition $S$ receives

The marginal contribution of player $i$ when joining coalition $S$ is:

<img src="img/formula1.png" width=150>

Now the question is: how do we aggregate this contribution across all possible configurations?

The simplest answer is to take the expectation over all possible teams.

$$\varphi_i(v) = \mathbb{E} \big[ v(S \cup \{i\}) - v(S) \big]$$

In order to compute expectation we need to define probabilistical model

There are two options. First one is the approach proposed by Shapley

### Method Idea
Shapley proposed to define sample space as all possible orderings of players and make each ordering equally probable

$$\varphi_i(v) = \frac{1}{n!} \sum_{R} \left[ v(P_i^R \cup \{i\}) - v(P_i^R) \right]$$

We iterate over all possible orderings of players.

- There are $n!$ ways to order a set of $n$ players
- We iterate over all those permutations
- For a given permutation, we find the position of player $i$
- All players to the left form the coalition that player $i$ is joining
- By iterating over all permutations, we are guaranteed to cover all possible coalitions



Here is an example for a 3-player coalition. We list all permutations and compute player $i$'s marginal contribution in each one:

<img src="img/formula3.png" width=350>

The Shapley Vector is the collection of Shapley Values for all players:
$\varphi = \{\varphi_1, \varphi_2 \ldots \varphi_N \}$


### Formula Variants

In most settings, the payoff doesn't depend on the order in which players join — only on *who* is in the coalition. We can exploit this to simplify: instead of summing over all $n!$ permutations, we group permutations that produce the same coalition.

The number of ways to order the players inside coalition $S$ is $|S|!$, and the number of ways to order the remaining players is $(n - |S| - 1)!$, so we get:

$$\varphi_{i}(v) = \sum_{S \subseteq N \setminus \{i\}} \frac{|S|! (n - |S| - 1)!}{n!} (v(S \cup \{i\}) - v(S))$$

Factoring out $1/n$, we can also write this using combinatorial notation:

$$\varphi_{i}(v) = \frac{1}{n} \sum_{S \subseteq N \setminus \{i\}} \binom{n - 1}{|S|}^{-1} (v(S \cup \{i\}) - v(S))$$

// Notice that contribution on the scale, is a part of game


# Axioms
Shapley suggested that his distribution is the best in some sense. To describe in which exactly way, he proposed 4 axioms.

Class of distributions follow 4 axioms:

### Efficiency

All the gain must be distributed:
$$\sum_{i \in N} \varphi_i(v) = v(N)$$

**Proof sketch:** Swap the order of summation — in each permutation, every player receives their marginal contribution, and these contributions sum to $v(N)$ by telescoping.

### Symmetry

If two players $i$ and $j$ are interchangeable — they make the same marginal contribution to every possible coalition — then they receive the same value:

$$\forall S \subseteq N \setminus \{i, j\}: \quad v(S \cup \{i\}) = v(S \cup \{j\}) \implies \varphi_i = \varphi_j$$

The method doesn't distinguish players by anything other than their actual contributions.

### Null Player (Dummy)

If a player adds nothing to any coalition, they receive zero:

$$\forall S \subseteq N: \quad v(S \cup \{i\}) = v(S) \implies \varphi_i = 0$$

### Additivity (Linearity)

For two independent games $v$ and $w$ played simultaneously:

$$\varphi_i(v + w) = \varphi_i(v) + \varphi_i(w)$$

You can solve sub-games independently and combine the results.

### Theorem
There exists a unique allocation rule satisfying all four axioms, and it is exactly the formula above.


## Other Properties


### Superadditivity

If the payoff function is superadditive — meaning it is always beneficial to form a coalition — then the payoff of any coalition is at least as large as the sum of its members' individual payoffs. As a consequence, each player's Shapley Value is at least as large as what they could earn alone.

### Shapley Value as Synergy

Shapley Value = sum of synergies per member across all coalitions

Informally, synergy occurs when the effect of a coalition exceeds the sum of individual effects (i.e. the value function is supermodular). Formally, synergy is the "uplift" gained by combining efforts. For two players:

$W_{X+Y} = V_{X+Y} - V_X - V_Y$

If $X$ and $Y$ are sub-teams rather than individual players, we extend this to multiple levels of inclusion:

$W_{X+Y} = V_{X+Y} - (V_{X} - V_{x_1} - V_{x_2}) - (V_{Y} - V_{y_1} - V_{y_2})$

<img src="img/synergy2.jpg" width=300>

Or for an arbitrary depth of nesting:

<img src="img/formula5.png" width=200>

The total synergy $w(S)$ is the sum of the uplifts across all subsets within $S$.

Synergy arises whenever players coordinate — it can be positive (coordination amplifies output) or negative (interference reduces it).

The Shapley Value for player $i$ equals the sum of synergies across all coalitions that include $i$, divided by the coalition size:

<img src="img/synergy1.png" width=200>

**Example — Manager and Workers:**

A team consists of a manager and a set of workers: $\{o, w_1 \ldots w_N\}$

Without the manager, the workers are idle:
$v(w_1) = \ldots = v(w_N) = 0$

Without the workers, the manager produces nothing:
$v(o) = 0$

Together, they deliver a result:
$v(o, w_1 \ldots w_N) = (N-1) \cdot p$

The synergy of any manager–worker pair is positive:
$w(o, w_1) = v(o, w_1) - v(o) - v(w_1)$

And the Shapley Value for each player reflects their share of those synergies across all possible coalitions.











## Shapley Value as a Marketing Attribution Algorithm

A customer visits your website after seeing a display ad on Monday, clicking a search result on Wednesday, and opening a promotional email on Friday — then makes a purchase. Which channel deserves the credit?

This is the attribution problem: given a sequence of marketing touchpoints that preceded a conversion, how do we distribute that conversion across the channels involved?

The question matters because attribution drives budget allocation. If search gets all the credit, you invest more in search. If the answer is wrong, so is the budget

## Why Naive Methods Fail

Most organizations start with a rule-based heuristic:

**First-touch** assigns 100% of credit to the first channel the customer interacted with. This rewards awareness-building channels (display, social) regardless of whether they actually influenced the final decision.

**Last-touch** assigns 100% to the final touchpoint before conversion. This is the most common default and systematically over-credits email and branded search — channels that capture intent already formed elsewhere.

**Linear** splits credit equally across all touchpoints. Simple and unbiased, but it treats a brief ad impression the same as a carefully read product comparison page.

**Time-decay** gives more credit to touchpoints closer to conversion. Intuitively appealing, but the decay rate is arbitrary and the method still ignores synergies between channels.

All of these share the same structural problem: they are rules invented before looking at the data. They cannot capture the fact that some channels work well together and poorly in isolation, or that the same channel may be effective in one combination and redundant in another.

## The Shapley Formulation

Shapley Value reframes attribution as a cooperative game:

- **Players** = marketing channels (search, display, email, social, affiliate, …)
- **Coalition** = any subset of channels that were part of a customer journey
- **Payoff** $v(S)$ = the conversion rate (or expected revenue) of journeys that included exactly the channels in $S$

The Shapley Value $\phi_i$ for channel $i$ is then its average marginal contribution across all possible channel combinations — exactly the permutation formula from game theory, applied here to conversion data.

The marginal contribution of channel $i$ to coalition $S$ is:

$$v(S \cup \{i\}) - v(S)$$

the difference in conversion rate between journeys that included channels $S \cup \{i\}$ versus journeys that included only $S$. The Shapley Value averages this across all coalitions $S \subseteq N \setminus \{i\}$, weighted by the standard factorial coefficient.

---

## What This Captures That Rules Don't

The key property is that Shapley attribution is **data-driven and interaction-aware**.

Consider a channel like branded email. In isolation it may have a high conversion rate — but that's because it reaches customers who were already going to convert. When you compute its marginal contribution to each coalition, you find it adds little to journeys that already included retargeting. Its Shapley Value reflects this: it gets credit for the customers it genuinely moved, not for the ones it merely touched.

Conversely, display advertising typically shows low last-touch conversion but high marginal contribution to journeys that later include search — it seeds awareness that other channels harvest. Shapley correctly attributes upstream value to it.

The **Efficiency axiom** guarantees that Shapley attribution is a proper distribution: the values across all channels sum to the total conversion, with nothing double-counted and nothing lost.

---

## Computing It in Practice

In theory, computing Shapley Values requires observing $v(S)$ for all $2^n$ channel subsets. With $n = 10$ channels this is 1024 subsets, which is manageable if your data volume is sufficient to estimate conversion rates reliably for each combination.

In practice two problems arise.

**Data sparsity:** many coalitions are rarely or never observed. A customer journey involving exactly {display, affiliate, email} without search may appear only a handful of times in the data, making $v(S)$ for that coalition a noisy estimate. Standard approaches include smoothing, combining similar coalitions, or restricting the analysis to the most common channel combinations.

**Channel ordering:** the game-theoretic formulation treats coalitions as sets, not sequences. But marketing touchpoints have order — a display impression before a search click is not the same customer journey as the reverse. Some implementations ignore order entirely; others define $v(S)$ on ordered sequences, which increases the complexity substantially. There is no universally agreed solution, and the choice affects results.

---

## Comparison to Data-Driven Alternatives

Shapley attribution is sometimes contrasted with **algorithmic attribution** based on logistic regression or survival models — where you fit a model to the full journey data and read off feature importances or coefficients for each channel.

The practical difference: regression-based methods can incorporate journey order, timing, and customer-level features naturally. Shapley is simpler to explain to a marketing team ("each channel gets its average marginal contribution") and has the uniqueness guarantee from the axioms — no arbitrary modeling choices about functional form.

A useful framing: Shapley attribution is the right default when you want a principled, explainable baseline with minimal modeling assumptions. Regression-based methods are worth the added complexity when journey structure (order, timing, frequency) is important and the data volume supports it.

In recent years **SHAP applied to a conversion model** has become a practical middle ground: fit a gradient boosted model on journey-level features, then use TreeSHAP to attribute each predicted conversion to individual channel features. This inherits the modeling flexibility of machine learning and the theoretical guarantees of Shapley — at the cost of requiring a well-calibrated model as an intermediate step.

## Banzhaf Value

The Banzhaf Value is an alternative way to measure a player's contribution in a cooperative game, proposed by John Banzhaf in 1965

__Idea:__ let's look at the marginal contribution of player $i$ when joining coalition $S$. but lets average differently — instead of averaging over permutations, we average uniformly over all subsets:

$$\beta_i(v) = \frac{1}{2^{n-1}} \sum_{S \subseteq N \setminus \{i\}} \left[ v(S \cup \{i\}) - v(S) \right]$$

Every subset $S \subseteq N \setminus \{i\}$ gets equal weight $\frac{1}{2^{n-1}}$, regardless of size. Shapley, by contrast, up-weights small and large coalitions relative to medium-sized ones — a direct consequence of the factorial coefficient.

How It Differs from Shapley<br>

The core difference is in what counts as "equally likely."

Shapley assumes players arrive in a random order, with every ordering equally probable. This means mid-sized coalitions appear less often than small or large ones.

Banzhaf assumes each player independently participates with probability 1/2. All $2^{n-1}$ subsets are then equally likely.

The practical consequence: Banzhaf Value **does not satisfy Efficiency** — the values don't necessarily sum to $v(N)$. This is a fundamental distinction. Banzhaf doesn't measure a player's *share of the pie*; it measures their **absolute power**

Axiomatics<br>

Banzhaf satisfies three of Shapley's four axioms — Symmetry, Null Player, and a modified version of Additivity — but not Efficiency.

In its place, Banzhaf adopts a different principle: **a player's marginal contribution is independent of what other players do**. This makes it more "individualistic" — it captures isolated power rather than a fair share of collective output

Application: Banzhaf Index in Voting<br>

The original application was measuring political power in weighted voting systems. A player is *critical* to coalition $S$ if their departure turns a winning coalition into a losing one:

$$v(S) = 1, \quad v(S \setminus \{i\}) = 0$$

The Banzhaf index is the fraction of coalitions in which a player is critical.

A classic example: in the UN Security Council, each of the 5 permanent members holds veto power. Despite there being 10 non-permanent members, their real power as measured by the Banzhaf index is far smaller than the 10:5 ratio would suggest

When to Use Banzhaf Instead of Shapley<br>

In the ML setting, Banzhaf is attractive because it's simpler to approximate — no coalition weighting, just average over random subsets. Some work (e.g. Karczmarz et al., 2022) suggests Banzhaf gives more stable estimates under sampling.

The downside is the lack of Efficiency: explanations don't sum to the prediction deviation, which makes them harder to interpret and communicate.



### Shapley Theorem

The theorem states: there exists a unique allocation rule satisfying all four axioms, and it is exactly the permutation formula. The proof has two parts — uniqueness and existence — and the uniqueness direction is the interesting one.

### Theorem Proof

#### Existence

Just verify that the formula satisfies all four axioms. This is mechanical: efficiency follows from a telescoping sum over any permutation, symmetry and null player follow directly from the definition, additivity follows from linearity of the sum. Nothing deep here.

#### Uniqueness

We need to show that the four axioms *force* the formula — that no other allocation rule can satisfy all four simultaneously
#### Step 1
Basis games

> Let's define a family of simple games called *unanimity games*. For each coalition $T \subseteq N$, define:
$$u_T(S) = \begin{cases} 1 & \text{if } T \subseteq S \\ 0 & \text{otherwise} \end{cases}$$
This game pays 1 if and only if the coalition contains all members of $T$, and 0 otherwise. The key fact is that unanimity games form a **basis**: any game $v$ can be written as a unique linear combination of unanimity games:
$$v = \sum_{T \subseteq N} c_T \cdot u_T$$
where the coefficients $c_T$ are determined by inclusion-exclusion:
$$c_T = \sum_{S \subseteq T} (-1)^{|T| - |S|} v(S)$$
These coefficients are exactly the synergy values we discussed earlier — the Möbius transform of $v$.


<br>

##### Step 2: Pin down the allocation on each basis game<br>

> Take any unanimity game $u_T$. Who contributes to it?
Any player $i \notin T$ adds nothing to any coalition — because $u_T(S \cup \{i\}) = u_T(S)$ for all $S$ when $i \notin T$. By the **Null Player axiom**, $\phi_i(u_T) = 0$.
The remaining players $i \in T$ are completely interchangeable — because $u_T$ treats all members of $T$ identically. By the **Symmetry axiom**, they must all receive the same value $\lambda$.
By the **Efficiency axiom**, the values must sum to $u_T(N) = 1$, so $|T| \cdot \lambda = 1$, giving $\lambda = 1/|T|$.
So on any unanimity game $u_T$, the axioms force:
$$\phi_i(u_T) = \begin{cases} 1/|T| & \text{if } i \in T \\ 0 & \text{if } i \notin T \end{cases}$$
There is no freedom here — the axioms uniquely determine the allocation on every basis game.

##### Step 3: Extend to all games via Additivity

> As we mentioned at step 1, any game can be represented as a sum $v = \sum_T c_T \cdot u_T$<br><br>By the **Additivity axiom**:
$$\phi_i(v) = \sum_{T \subseteq N} c_T \cdot \phi_i(u_T) = \sum_{T \ni i} \frac{c_T}{|T|}$$
This is uniquely determined. No choices were made — the axioms dictate every step.

##### Step 4: Verify this equals the permutation formula

> Substituting the expression for $c_T$ and rearranging the sum (grouping by which coalition $T$ appears in which permutation) recovers the permutation formula exactly. This step is algebraically tedious but conceptually straightforward — it's the same sum, just reindexed

Theorem proved

### Why the Nobel Prize?

The prize rested primarily on the **Gale-Shapley algorithm** (1962), developed jointly with David Gale. The problem: given a set of men and a set of women, each with ranked preferences over the other group, find a *stable matching* — one where no man and woman both prefer each other to their current partner.

Gale and Shapley proved that a stable matching always exists, and gave a simple deferred-acceptance algorithm that finds one. This was a foundational result in matching theory, and Alvin Roth spent decades applying it to real markets: matching medical residents to hospitals, students to schools, kidney donors to recipients. The practical impact was enormous and concrete.

The Shapley Value was part of Shapley's broader body of work cited by the committee — along with stochastic games, the Shapley-Shubik power index, and contributions to core theory — but it was not the singular reason for the prize.

# SHAP
---

SHAP stands for "**SH**apley **A**dditive ex**P**lanations" and is a Python library for explaining the predictions of machine learning models. As the name suggests, it applies Shapley Value mechanics to the prediction problem:

- **Game** = a single prediction at point $x$ using a pretrained model $f$
- **Players** = features
- **Coalition** = the subset of features "available" to the model
- **Payoff** = the prediction's deviation from the baseline $\mathbb{E}[f]$

Just as Shapley Value decomposes a team's payoff into individual contributions, SHAP decomposes a model's prediction into a sum of per-feature contributions:

<img src="img/formula.png" width=300>

where $\phi_0 = \mathbb{E}[f(x)]$ is the baseline (the model's average prediction over the dataset), and each $\phi_i$ tells us how much feature $i$ pushed the prediction up or down for this particular instance.

The challenge is that computing exact SHAP values requires evaluating the model on all $2^n$ feature subsets — which is exponential. Two main algorithms address this: **KernelSHAP** (model-agnostic) and **TreeSHAP** (specialized for tree-based models)

## KernelSHAP
Let's explore some blackbox model $f(x)$. Suppose we are given some point $x$

Algorithm
1. Generate a pool of random feature subsets $\{S\}$, for example $S_i = (0,1,0,0,1,0)$
2. For each subset compute the model output $f(x)$<br>this is done by pluging in known features and averaging over unkown features $f_S(x) = E_{x \sim D}[f(x|x_S)]$<br>proper averaging is expensive, in practice we usually just impute with random values drawn from feature distribution
3. We know that for any subset the output is decomposed into sum of Shapley coefficients<br>$f(x) = \phi_0 + \phi_1 + \phi_2 + \phi_n$<br>To get values let's solve a regression task<br><Br>$\begin{cases} f(x|x_1,x_3)=\phi_0 + \phi_1 + 0 + \phi_3 \\ f(x|x_3)=\phi_0 + 0 + 0 + \phi_3 \\ ... \\ f(x|x_2)=\phi_0 + 0 + \phi_2 + 0 \end{cases}$<br><br>
5. If we Make regression weighted observation by $\pi(z)$ it gonna give exactly Shapley values
$$\pi(z) = \frac{n - 1}{\binom{n}{|z|} |z| (n - |z|)}$$

As optimization problem:
$$\min_\phi \sum_{z \in \{0,1\}^n} \pi(z) \left[f(z) - g(z)\right]^2 \quad \text{where} \quad g(z) = \phi_0 + \sum_j \phi_j s_i$$

__Weighting intuition__<br>
Small and large coalitions receive high weight; medium-sized coalitions receive low weight. This directly reflects the structure of the Shapley formula — small and large coalitions appear in the vast majority of permutations.

__Why Kernel in the method name?__<br> Because we weight observations using kernels (like in Kernel Rgression for example)



Complexity: $O(m \cdot n \cdot |\mathcal{D}|)$. In practice, $m \approx 2n + 2048$ gives a good approximation. For exact results you'd need $m = 2^n$ — still exponential.

In code:<br>`shap.KernelExplainer(model.predict, background_data)`

✗ Does not account for feature correlations — "absent" features are replaced by independent background samples

## TreeSHAP

In a decision tree the structure makes it possible to compute exact SHAP values analytically — no sampling required.

Naive traversal:
1. For each feature
2.     For each subset S
3.         compute value averaged over unused features

Idea of TreeSHAP is how to compute all data in one loop

When some features are "absent" (not in coalition $S$), we need to estimate what the model would predict if it couldn't see those features. TreeSHAP does this by tracking how training samples flow through the tree:

This gives us $\mathbb{E}[f(x) \mid x_S]$ — the expected prediction conditioned on the known features — computed analytically without enumerating all $2^n$ subsets.

### The Algorithm

TreeSHAP traverses the tree top-down, maintaining for each node a polynomial weight vector $(p_0, p_1, \ldots, p_d)$, where $p_k$ is the probability that the instance ends up in this node when exactly $k$ of the $d$ features on the path so far belong to coalition $S$.

By accumulating these weights across all leaves, TreeSHAP computes the expected prediction for every possible coalition in a single pass — without iterating over subsets explicitly.

**Complexity:** $O(T \cdot L \cdot D^2)$ where $T$ = number of trees, $L$ = number of leaves, $D$ = max depth.

For a typical gradient boosting setup ($T = 1000$, $D = 6$, $L \approx 64$): SHAP values are computed in milliseconds per instance.

✓ **Exact** — not an approximation  
✓ **Fast** — polynomial in tree parameters, not exponential in features  
✓ Accounts for feature correlations as captured by the tree structure  
✗ Only works for tree-based models

### Conditional vs. Interventional

There is an important conceptual distinction between the two flavors of TreeSHAP:

**Conditional (default):** Uses the training distribution in tree nodes to estimate $\mathbb{E}[f(x) \mid x_S]$. This respects feature correlations — "absent" features are imputed according to what is realistic given the observed ones.

**Interventional:** Replaces absent features with independent samples from a background dataset $\mathcal{D}$, estimating $\mathbb{E}_{x_{\bar{S}}}[f(x_S, x_{\bar{S}})]$. This can produce unrealistic feature combinations when features are correlated, but satisfies the Null Player axiom more cleanly.

The choice matters for interpretation:
- Use **interventional** to explain the model's behavior (how does the model respond to feature $i$, independent of other features?)
- Use **conditional** to explain the data-generating process (what is the true role of feature $i$ in the real phenomenon?)

---

## KernelSHAP vs. TreeSHAP — Summary

| | KernelSHAP | TreeSHAP |
|---|---|---|
| **Model type** | Any | Trees / ensembles |
| **Accuracy** | Approximate | Exact |
| **Complexity** | $O(m \cdot n \cdot |\mathcal{D}|)$ | $O(T \cdot L \cdot D^2)$ |
| **Marginalization** | Interventional | Conditional (default) |
| **Feature correlations** | Ignored | Partially handled |
| **Speed** | Slow for $n > 100$ | Milliseconds |
| **Code** | `shap.KernelExplainer` | `shap.TreeExplainer` |

---



KernelSHAP works with any model but is slow. TreeSHAP is fast and exact but tree-specific. Neural networks call for a different approach — one that exploits what they have and trees don't: **gradients**.

### DeepSHAP

DeepSHAP (2017) is built on top of **DeepLIFT** (Shrikumar et al., 2017) and adapts it toward Shapley Values.

Contribution of $x_i$ = how change in x influences y

- We choose initial point (could be all zeros)
- $\Delta y = f(x) - f(x_{ref})$
- For each neuron
- 

$$\phi_i \approx (x_i - \bar{x}_i) \cdot \mathbb{E}\left[\frac{\partial f}{\partial x_i}\right]$$

`shap.DeepExplainer`

### GradientSHAP

GradientSHAP combines **Integrated Gradients** (Sundararajan et al., 2017) with the Shapley sampling idea.

Integrated Gradients is one of the most theoretically grounded attribution methods for neural networks. It integrates the gradient along a straight path from baseline $\bar{x}$ to input $x$:

$$\text{IG}_i(x) = (x_i - \bar{x}_i) \int_0^1 \frac{\partial f(\bar{x} + \alpha(x - \bar{x}))}{\partial x_i} \, d\alpha$$

GradientSHAP adds a Shapley twist: instead of a single fixed baseline, it samples **random baselines from a background dataset** and averages the integrated gradients. This pushes the result closer to true SHAP values when the baseline distribution matters.

In practice the integral is approximated via Monte Carlo: sample a few random points along the path from $\bar{x}$ to $x$ and average the gradients there. Implemented as `shap.GradientExplainer`.

Comparison<br>

DeepSHAP is faster — one backward pass. GradientSHAP is slower but theoretically closer to true SHAP. Both depend on baseline choice, which in practice often matters more than the choice between the two methods. For images, a black image or blurred image makes a reasonable baseline; for tabular data, a random background sample is typical.

## The Core

Shapley Value asks: how do we fairly distribute the payoff? The Core asks a different question:<br> under what allocation would no group of players prefer to leave the grand coalition and play on their own?

This is a question about stability, not fairness.

Definition<br>

An allocation $(x_1, \dots, x_n)$ is an element of the **Core** if two conditions hold.

Efficiency:<br>
$\sum_{i \in N} x_i = v(N)$

Coalition rationality — for every coalition $S \subseteq N$:<br>
$\sum_{i \in S} x_i \geq v(S)$

The second condition is the key one: no subgroup receives less than it could earn by breaking away. If this is violated for some coalition $S$, the players in $S$ have a rational incentive to defect.

The Problem with the Core<br>

The Core can be empty. For many games, no allocation simultaneously satisfies all coalitions. The classic example is a majority-voting game with three equal players: any two can outvote the third, so no stable allocation exists.

When non-empty, the Core is typically not a single point but a multi-dimensional set — it says "anywhere in here is safe," but offers no unique answer

Relationship to Shapley Value

For **convex games** (where $v(S \cup T) + v(S \cap T) \geq v(S) + v(T)$) the Core is always non-empty, and the Shapley Value always lies inside it. This is an important result: it means the Shapley allocation is not only *fair* by axioms, but also **stable** — no subgroup wants to defect.

In ML settings, $v(S) = \mathbb{E}[f(x) \mid x_S]$ is generally not convex, so the Core may be empty and its relationship to SHAP values is non-trivial. In marketing attribution, however, where conversion functions often exhibit diminishing returns, convexity frequently holds.

## Plots

**Force plot** — starts from $\mathbb{E}[f]$ and shows how each feature pushes the prediction up (red) or down (blue), arriving at the final prediction $f(x)$.

<img src="img/shap1.png" width=500>

**Beeswarm plot** — for each feature (vertical axis), shows its SHAP value across all data points (horizontal axis). Color encodes the feature's value. This gives a global view of feature importance and directionality.

<img src="img/shap2.png" width=500>

**Mean |SHAP| bar chart** — the average absolute SHAP value per feature, giving a clean global feature importance ranking.

<img src="img/shap3.png" width=400>

**Waterfall plot** — shows the breakdown for a single prediction: starting from the global baseline $\mathbb{E}[f]$, each feature's contribution is shown as a step, ending at the model's actual output $f(x)$.

<img src="img/shap4.png" width=750>

# SHAP Dependence Plot

What It Is<br>

A SHAP Dependence Plot is a scatter plot where:
- X-axis: the value of feature $x_i$
- Y-axis: the SHAP value $\phi_i$ for that feature
- Each point: one observation from the dataset

Why It's Better than PDP<br>

Partial Dependence Plots fix all other features at their mean values and vary $x_i$, reading off the model's predicted response. The problem: averaging across the full dataset conceals heterogeneity. If the effect of $x_i$ is positive for half the dataset and negative for the other half, the PDP reports approximately zero — and hides the story entirely.

SHAP Dependence Plots show each observation separately, making it possible to see nonlinear effects, threshold behavior, and how widely the effect of a feature varies across individuals.

Coloring by Interaction<br>

SHAP automatically selects the feature $j$ that most strongly interacts with $i$ (via SHAP Interaction Values) and colors each point by the value of $x_j$. If points of one color are systematically higher or lower, that's a visible interaction signal.

Example: feature "temperature" on the X-axis, SHAP on the Y-axis, points colored by "humidity." If high-humidity points cluster above low-humidity points at high temperatures, the model has learned a temperature × humidity interaction

Relationship to ICE Plots<br>

Individual Conditional Expectation (ICE) plots are conceptually close: for each observation, draw a separate curve showing how the prediction changes as $x_i$ varies, with all other features held at their actual values. PDP is just the average of all ICE curves.

SHAP Dependence Plots are not the same as ICE — they show one point per observation rather than a full curve. But they add something ICE can't: a clean separation of feature $i$'s contribution from the rest of the prediction.

## SHAP Interaction Values

The Limitation of Scalar SHAP<br>

A standard SHAP value $\phi_i$ bundles together two distinct things: the *direct effect* of feature $i$, and its *interaction effects* with other features. These can't be separated from a single number.

For example, "age" and "physical activity" may jointly produce an effect that neither explains alone — exercise might reduce risk for younger individuals but have the opposite association for older ones. Scalar SHAP will average over this heterogeneity and lose it

## Definition<br>

SHAP Interaction Values form a matrix $\Phi \in \mathbb{R}^{n \times n}$, where off-diagonal entries are defined as:

$$\Phi_{ij} = \sum_{S \subseteq N \setminus \{i,j\}} \frac{|S|!(n-|S|-2)!}{2(n-1)!} \delta_{ij}(f, x, S)$$

where $\delta_{ij}$ is a discrete second derivative:

$$\delta_{ij} = f(x_{S \cup \{i,j\}}) - f(x_{S \cup \{i\}}) - f(x_{S \cup \{j\}}) + f(x_S)$$

This is exactly the intuitive definition of interaction: how much does the joint contribution of $i$ and $j$ exceed the sum of their individual contributions?

Diagonal entries $\Phi_{ii}$ capture the *main effect* of feature $i$ with interactions removed.

## Properties<br>

The matrix is symmetric: $\Phi_{ij} = \Phi_{ji}$.

Row sums recover standard SHAP: $\phi_i = \sum_j \Phi_{ij}$.

The full matrix sums to the prediction deviation: $\sum_{i,j} \Phi_{ij} = f(x) - \mathbb{E}[f]$.

## Computing and Interpreting<br>

For tree models, TreeSHAP computes exact interaction values in $O(T L D^2)$ — the same complexity as standard SHAP. Call: `shap.TreeExplainer(model).shap_interaction_values(X)`. For arbitrary models, exact computation requires $O(3^n)$ model calls, making it practical only with trees.

Two useful visualizations fall out of the interaction matrix. A **dependence plot colored by interaction**: plot $x_i$ vs $\phi_i$, and color each point by the value of the most-interacting feature $x_j$ — this reveals how $i$'s effect shifts as $j$ changes. An **interaction heatmap**: average $|\Phi_{ij}|$ across the dataset and display the full $n \times n$ matrix to get a global picture of where the model's interactions live.





## Practical Pitfalls

1. Background Dataset Choice Changes the Answer<br>

KernelSHAP, interventional TreeSHAP, and DeepSHAP all require a background dataset $\mathcal{D}$ for marginalizing absent features. SHAP values for the same instance can **change substantially** depending on what's in $\mathcal{D}$.

This isn't a bug — it's a consequence of the fact that SHAP answers the question "deviation from what?" A background of 100 random training rows gives one answer; a background restricted to a specific subpopulation gives a different one, because it's a different question.

Practical rule: choose the background deliberately, document it, and treat SHAP values from different backgrounds as answering different questions. A common default is a k-means summary of the training set ($k \approx 100$).

2. Feature Correlations Undermine Interpretation<br>

With interventional marginalization, SHAP substitutes random background values into absent features. When features are correlated, this produces unrealistic inputs — combinations that don't exist in nature. The model may behave unpredictably on these inputs, and the resulting SHAP values reflect that extrapolation, not the feature's genuine contribution.

Example: "height" and "weight" are strongly correlated. Interventional SHAP may evaluate the model at height = 190 cm, weight = 45 kg. If the model was trained only on realistic data, its output at that point is extrapolation.

Conditional TreeSHAP partially addresses this by using the training distribution within tree nodes — but if the tree rarely saw a particular combination, even that node-level statistics are unreliable.

3. Explaining the Model ≠ Explaining the World<br>

SHAP explains **why the model produces a particular output** — not why the underlying phenomenon works the way it does. These are different questions, and conflating them is the most consequential mistake in practice.

If a model was trained on historically biased data (hiring decisions, credit scoring), SHAP will faithfully explain the logic of that biased model. Precisely because it is faithful, the explanation can be mistaken for a description of reality.

Additionally, when features are correlated, SHAP may concentrate all attribution on one feature (whichever the tree splits on first) even if another is the true causal driver. **SHAP does not answer causal questions.**

4. KernelSHAP Is Noisy<br>

KernelSHAP uses Monte Carlo sampling. With small $m$, estimates are unstable: two runs on the same instance produce different values. This is especially pronounced for features with small SHAP values near zero — the relative estimation error can be large even when the absolute error is modest.

In practice: fix the random seed, increase $m$ for explanations that matter, and don't interpret small differences between SHAP values as meaningful signal.

5. SHAP Is a Debugging Tool, Not an Audit<br>

SHAP works well as a tool for the model developer: understanding where the model is "looking," identifying dominant features, catching data leakage. For regulatory auditing or fairness assessment it is not sufficient on its own — it doesn't establish causality, doesn't remove bias, and its interpretation depends on methodological choices (background, marginalization type) that are rarely disclosed.

This doesn't make SHAP useless in regulatory contexts, but it means knowing precisely what it measures — and what it doesn't.

# Usage Example
---

In [ ]:
import shap
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Load data
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, random_state=42
)

# Train a tree-based model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# --- TreeSHAP ---
# TreeExplainer works with tree-based models and is exact + fast
tree_explainer = shap.TreeExplainer(model)
shap_values = tree_explainer.shap_values(X_test)

# shap_values is a list of arrays, one per class
# Explain the first test instance for class 0
shap.initjs()
shap.force_plot(
    tree_explainer.expected_value[0],
    shap_values[0][0],
    X_test[0],
    feature_names=iris.feature_names
)

In [ ]:
# Global feature importance — beeswarm plot (class 0 vs rest)
shap.summary_plot(shap_values[0], X_test, feature_names=iris.feature_names)

In [ ]:
# --- KernelSHAP ---
# KernelExplainer works with ANY model — only needs a callable predict function
# We pass a small background dataset (summarized with k-means for speed)
background = shap.kmeans(X_train, 10)  # summarize background in 10 clusters

kernel_explainer = shap.KernelExplainer(model.predict_proba, background)

# Compute SHAP values for a small subset (KernelSHAP is slower)
kernel_shap_values = kernel_explainer.shap_values(X_test[:10])

# Force plot for the first instance, class 0
shap.force_plot(
    kernel_explainer.expected_value[0],
    kernel_shap_values[0][0],
    X_test[0],
    feature_names=iris.feature_names
)